In [176]:
# Data Transformation: CSV to Data Warehouse (OLAP)
# This notebook transforms raw CSV data into a star schema for the source database

In [177]:
# 1. Setup and Imports

In [178]:
import pandas as pd
import numpy as np
from datetime import datetime
import os
from sqlalchemy import create_engine, text
import glob

In [179]:
# 2. Database Connection to Source DB

In [180]:
# Database connection parameters (from your .env)
DB_USER = 'source_user'
DB_PASSWORD = 'source_pass'
DB_HOST = 'postgres-source'  # Service name from docker-compose
DB_PORT = '5432'
DB_NAME = 'tpch' # Old DB

In [181]:
# Create connection engine
engine = create_engine(f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

In [182]:
# Test connection
with engine.connect() as conn:
    result = conn.execute(text("SELECT 1"))
    print("✅ Connected to source database successfully!")

✅ Connected to source database successfully!


In [183]:
# 3. Load and Combine CSV Files

In [184]:
# Path to your data folder (with /raw subfolder)
data_path = '/home/jovyan/data/raw/'

# Find all CSV files
csv_files = glob.glob(f'{data_path}*.csv')
print(f"Found {len(csv_files)} CSV files:")
for file in csv_files:
    print(f"  - {os.path.basename(file)}")

# Load and combine all CSV files
# Here we added the source_file column so that we can trace our data
df_list = []
for file in csv_files:
    df = pd.read_csv(file)
    df['source_file'] = os.path.basename(file)  # Add source filename
    df_list.append(df)
    print(f"Loaded {os.path.basename(file)}: {len(df)} rows")

# Combine all dataframes
raw_data = pd.concat(df_list, ignore_index=True)

# Combine all dataframes
raw_data = pd.concat(df_list, ignore_index=True)
print(f"\n✅ Combined data: {len(raw_data)} total rows")
print(f"Columns: {list(raw_data.columns)}")

Found 3 CSV files:
  - sales_2025_01.csv
  - sales_2025_02.csv
  - sales_2025_03.csv
Loaded sales_2025_01.csv: 20 rows
Loaded sales_2025_02.csv: 20 rows
Loaded sales_2025_03.csv: 10 rows

✅ Combined data: 50 total rows
Columns: ['order_id', 'order_date', 'customer_id', 'customer_name', 'city', 'country', 'product_id', 'product_name', 'category', 'quantity', 'unit_price', 'unit_cost', 'source_file']


In [185]:
raw_data.head()

,order_id,order_date,customer_id,customer_name,city,country,product_id,product_name,category,quantity,unit_price,unit_cost,source_file
0,1001,2025-01-02,C001,Ahmed Hassan,Cairo,Egypt,P100,Laptop,Electronics,1,1200,900,sales_2025_01.csv
1,1002,2025-01-02,C002,Sara Ali,Giza,Egypt,P101,Mouse,Accessories,2,25,10,sales_2025_01.csv
2,1003,2025-01-03,C003,Omar Khaled,Alexandria,Egypt,P102,Keyboard,Accessories,1,45,20,sales_2025_01.csv
3,1004,2025-01-03,C004,Mona Youssef,Mansoura,Egypt,P103,Monitor,Electronics,1,300,220,sales_2025_01.csv
4,1005,2025-01-04,C005,Nour Ibrahim,Tanta,Egypt,P104,Printer,Office,1,180,140,sales_2025_01.csv


In [186]:
# 4. Data Cleaning and Validation

In [187]:
print("Data Info:")
raw_data.info()

print("\nSample Data:")
raw_data.head()

# Check for missing values
print("\nMissing Values:")
print(raw_data.isnull().sum())

# Convert date column
raw_data['order_date'] = pd.to_datetime(raw_data['order_date'])

# Remove any duplicates if needed
raw_data = raw_data.drop_duplicates()
print(f"\nAfter removing duplicates: {len(raw_data)} rows")

Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   order_id       50 non-null     int64 
 1   order_date     50 non-null     object
 2   customer_id    50 non-null     object
 3   customer_name  50 non-null     object
 4   city           50 non-null     object
 5   country        50 non-null     object
 6   product_id     50 non-null     object
 7   product_name   50 non-null     object
 8   category       50 non-null     object
 9   quantity       50 non-null     int64 
 10  unit_price     50 non-null     int64 
 11  unit_cost      50 non-null     int64 
 12  source_file    50 non-null     object
dtypes: int64(4), object(9)
memory usage: 5.2+ KB

Sample Data:

Missing Values:
order_id         0
order_date       0
customer_id      0
customer_name    0
city             0
country          0
product_id       0
product_name     0
category

In [188]:
# 5. Create Dimension Tables

In [189]:
# 5.1 Customer Dimension with Type 2 SCD
print("\n=== CREATING CUSTOMER DIMENSION (TYPE 2 SCD) ===")

# First, get the earliest and latest date for each customer-state combination
customer_history = []

# Group by customer_id and all attributes to track changes
customer_attributes = ['customer_id', 'customer_name', 'city', 'country']

# Sort by date to track changes over time
raw_data_sorted = raw_data.sort_values(['customer_id', 'order_date'])

# Track changes for each customer
for customer_id, group in raw_data_sorted.groupby('customer_id'):
    current_attrs = None
    start_date = None
    
    for idx, row in group.iterrows():
        attrs = tuple(row[['customer_name', 'city', 'country']].values)
        
        if current_attrs is None:
            # First occurrence
            current_attrs = attrs
            start_date = row['order_date']
        elif attrs != current_attrs:
            # Attribute change detected
            # End the previous version
            customer_history.append({
                'customer_id': customer_id,
                'customer_name': current_attrs[0],
                'city': current_attrs[1],
                'country': current_attrs[2],
                'valid_from': start_date,
                'valid_to': row['order_date'] - pd.Timedelta(days=1),  # Day before change
                'is_current': False
            })
            # Start new version
            current_attrs = attrs
            start_date = row['order_date']
    
    # Add the final version (still current)
    if current_attrs is not None:
        customer_history.append({
            'customer_id': customer_id,
            'customer_name': current_attrs[0],
            'city': current_attrs[1],
            'country': current_attrs[2],
            'valid_from': start_date,
            'valid_to': pd.NaT,  # No end date = current
            'is_current': True
        })

dim_customer = pd.DataFrame(customer_history)
dim_customer['customer_sk'] = range(1, len(dim_customer) + 1)

# Add source tracking (show which files contributed to this version)
source_tracking = raw_data.groupby(['customer_id', 'customer_name', 'city', 'country'])['source_file'].agg(lambda x: ', '.join(x.unique())).reset_index()
dim_customer = dim_customer.merge(
    source_tracking, 
    on=['customer_id', 'customer_name', 'city', 'country'], 
    how='left'
)
dim_customer.rename(columns={'source_file': 'source_files'}, inplace=True)

# Ensure correct data types
dim_customer['customer_id'] = dim_customer['customer_id'].astype(str)
dim_customer['customer_name'] = dim_customer['customer_name'].astype(str)
dim_customer['city'] = dim_customer['city'].astype(str)
dim_customer['country'] = dim_customer['country'].astype(str)
dim_customer['valid_from'] = pd.to_datetime(dim_customer['valid_from'])
dim_customer['valid_to'] = pd.to_datetime(dim_customer['valid_to'])
dim_customer['is_current'] = dim_customer['is_current'].astype(bool)

print(f"Customer Dimension: {len(dim_customer)} rows (including historical changes)")
print(f"  - Current customers: {dim_customer['is_current'].sum()}")
print(f"  - Historical versions: {len(dim_customer) - dim_customer['is_current'].sum()}")
print(dim_customer.dtypes)
dim_customer.head(10)


=== CREATING CUSTOMER DIMENSION (TYPE 2 SCD) ===
Customer Dimension: 38 rows (including historical changes)
  - Current customers: 36
  - Historical versions: 2
customer_id              object
customer_name            object
city                     object
country                  object
valid_from       datetime64[ns]
valid_to         datetime64[ns]
is_current                 bool
customer_sk               int64
source_files             object
dtype: object


,customer_id,customer_name,city,country,valid_from,valid_to,is_current,customer_sk,source_files
0,C001,Ahmed Hassan,Cairo,Egypt,2025-01-02,2025-02-28,False,1,"sales_2025_01.csv, sales_2025_02.csv"
1,C001,Ahmed Hassan,Giza,Egypt,2025-03-01,NaT,True,2,sales_2025_03.csv
2,C002,Sara Ali,Giza,Egypt,2025-01-02,NaT,True,3,"sales_2025_01.csv, sales_2025_03.csv"
3,C003,Omar Khaled,Alexandria,Egypt,2025-01-03,NaT,True,4,"sales_2025_01.csv, sales_2025_02.csv, sales_20..."
4,C004,Mona Youssef,Mansoura,Egypt,2025-01-03,2025-03-02,False,5,sales_2025_01.csv
5,C004,Mona Youssef,Damietta,Egypt,2025-03-03,NaT,True,6,sales_2025_03.csv
6,C005,Nour Ibrahim,Tanta,Egypt,2025-01-04,NaT,True,7,sales_2025_01.csv
7,C006,Yara Adel,Cairo,Egypt,2025-01-05,NaT,True,8,"sales_2025_01.csv, sales_2025_02.csv, sales_20..."
8,C007,Mostafa Samir,Giza,Egypt,2025-01-05,NaT,True,9,sales_2025_01.csv
9,C008,Huda Nabil,Alexandria,Egypt,2025-01-06,NaT,True,10,sales_2025_01.csv


In [190]:
# 5.2 Product Dimension with Type 2 SCD and Source Attribution
print("\n=== CREATING PRODUCT DIMENSION (TYPE 2 SCD) ===")

# Track product changes over time
product_history = []
product_attributes = ['product_id', 'product_name', 'category', 'unit_price', 'unit_cost']

# Sort by date to track changes
raw_data_sorted = raw_data.sort_values(['product_id', 'order_date'])

for product_id, group in raw_data_sorted.groupby('product_id'):
    current_attrs = None
    start_date = None
    
    for idx, row in group.iterrows():
        attrs = tuple(row[['product_name', 'category', 'unit_price', 'unit_cost']].values)
        
        if current_attrs is None:
            current_attrs = attrs
            start_date = row['order_date']
        elif attrs != current_attrs:
            # Attribute change detected
            product_history.append({
                'product_id': product_id,
                'product_name': current_attrs[0],
                'category': current_attrs[1],
                'unit_price': current_attrs[2],
                'unit_cost': current_attrs[3],
                'valid_from': start_date,
                'valid_to': row['order_date'] - pd.Timedelta(days=1),
                'is_current': False
            })
            current_attrs = attrs
            start_date = row['order_date']
    
    # Add final version
    if current_attrs is not None:
        product_history.append({
            'product_id': product_id,
            'product_name': current_attrs[0],
            'category': current_attrs[1],
            'unit_price': current_attrs[2],
            'unit_cost': current_attrs[3],
            'valid_from': start_date,
            'valid_to': pd.NaT,
            'is_current': True
        })

dim_product = pd.DataFrame(product_history)
dim_product['product_sk'] = range(1, len(dim_product) + 1)
dim_product['profit_margin'] = dim_product['unit_price'] - dim_product['unit_cost']

# Add source tracking for products
source_tracking_product = raw_data.groupby(['product_id', 'product_name', 'category', 'unit_price', 'unit_cost'])['source_file'].agg(lambda x: ', '.join(x.unique())).reset_index()
dim_product = dim_product.merge(
    source_tracking_product,
    on=['product_id', 'product_name', 'category', 'unit_price', 'unit_cost'],
    how='left'
)
dim_product.rename(columns={'source_file': 'source_files'}, inplace=True)

# Ensure correct data types
dim_product['product_id'] = dim_product['product_id'].astype(str)
dim_product['product_name'] = dim_product['product_name'].astype(str)
dim_product['category'] = dim_product['category'].astype(str)
dim_product['unit_price'] = dim_product['unit_price'].astype(float)
dim_product['unit_cost'] = dim_product['unit_cost'].astype(float)
dim_product['profit_margin'] = dim_product['profit_margin'].astype(float)
dim_product['valid_from'] = pd.to_datetime(dim_product['valid_from'])
dim_product['valid_to'] = pd.to_datetime(dim_product['valid_to'])
dim_product['is_current'] = dim_product['is_current'].astype(bool)

print(f"Product Dimension: {len(dim_product)} rows (including historical changes)")
print(f"  - Current products: {dim_product['is_current'].sum()}")
print(f"  - Historical versions: {len(dim_product) - dim_product['is_current'].sum()}")
print(dim_product.dtypes)
dim_product.head(10)


=== CREATING PRODUCT DIMENSION (TYPE 2 SCD) ===
Product Dimension: 38 rows (including historical changes)
  - Current products: 24
  - Historical versions: 14
product_id               object
product_name             object
category                 object
unit_price              float64
unit_cost               float64
valid_from       datetime64[ns]
valid_to         datetime64[ns]
is_current                 bool
product_sk                int64
profit_margin           float64
source_files             object
dtype: object


,product_id,product_name,category,unit_price,unit_cost,valid_from,valid_to,is_current,product_sk,profit_margin,source_files
0,P100,Laptop,Electronics,1200.0,900.0,2025-01-02,2025-01-31,False,1,300.0,sales_2025_01.csv
1,P100,Laptop,Electronics,1180.0,900.0,2025-02-01,2025-02-28,False,2,280.0,sales_2025_02.csv
2,P100,Laptop Pro,Electronics,1300.0,980.0,2025-03-01,NaT,True,3,320.0,sales_2025_03.csv
3,P101,Mouse,Accessories,25.0,10.0,2025-01-02,NaT,True,4,15.0,"sales_2025_01.csv, sales_2025_02.csv, sales_20..."
4,P102,Keyboard,Accessories,45.0,20.0,2025-01-03,NaT,True,5,25.0,"sales_2025_01.csv, sales_2025_02.csv"
5,P103,Monitor,Electronics,300.0,220.0,2025-01-03,NaT,True,6,80.0,"sales_2025_01.csv, sales_2025_02.csv"
6,P104,Printer,Office,180.0,140.0,2025-01-04,2025-02-01,False,7,40.0,sales_2025_01.csv
7,P104,Printer,Office,185.0,140.0,2025-02-02,NaT,True,8,45.0,sales_2025_02.csv
8,P105,Desk Chair,Furniture,95.0,70.0,2025-01-04,NaT,True,9,25.0,"sales_2025_01.csv, sales_2025_02.csv"
9,P106,USB Cable,Accessories,8.0,3.0,2025-01-05,NaT,True,10,5.0,"sales_2025_01.csv, sales_2025_02.csv"


In [191]:
# 5.3 Date Dimension
# First convert order_date to datetime
raw_data['order_date'] = pd.to_datetime(raw_data['order_date'])

dates = raw_data['order_date'].drop_duplicates().reset_index(drop=True)

dim_date = pd.DataFrame({
    'date_sk': range(1, len(dates) + 1),
    'date': dates,
    'year': dates.dt.year.astype(int),
    'month': dates.dt.month.astype(int),
    'day': dates.dt.day.astype(int),
    'quarter': dates.dt.quarter.astype(int),
    'day_of_week': dates.dt.dayofweek.astype(int),
    'day_name': dates.dt.day_name().astype(str),
    'month_name': dates.dt.month_name().astype(str),
    'is_weekend': (dates.dt.dayofweek >= 5).astype(bool)
})

print(f"Date Dimension: {len(dim_date)} unique dates")
print(dim_date.dtypes)
dim_date.head()

Date Dimension: 25 unique dates
date_sk                 int64
date           datetime64[ns]
year                    int64
month                   int64
day                     int64
quarter                 int64
day_of_week             int64
day_name               object
month_name             object
is_weekend               bool
dtype: object


,date_sk,date,year,month,day,quarter,day_of_week,day_name,month_name,is_weekend
0,1,2025-01-02,2025,1,2,1,3,Thursday,January,False
1,2,2025-01-03,2025,1,3,1,4,Friday,January,False
2,3,2025-01-04,2025,1,4,1,5,Saturday,January,True
3,4,2025-01-05,2025,1,5,1,6,Sunday,January,True
4,5,2025-01-06,2025,1,6,1,0,Monday,January,False


In [192]:
# Check Customer Dimension for duplicates
print("=== CUSTOMER DIMENSION ANALYSIS ===")
customer_dupes = dim_customer.groupby('customer_id').size().reset_index(name='occurrences')
problem_customers = customer_dupes[customer_dupes['occurrences'] > 1]

if len(problem_customers) > 0:
    print(f"⚠️ Found {len(problem_customers)} customer_ids with multiple rows!")
    for _, row in problem_customers.head(5).iterrows():
        cust_id = row['customer_id']
        print(f"\nCustomer ID {cust_id} appears {row['occurrences']} times:")
        display(dim_customer[dim_customer['customer_id'] == cust_id])
else:
    print("✅ All customer_ids are unique")

# Check Product Dimension for duplicates
print("\n=== PRODUCT DIMENSION ANALYSIS ===")
product_dupes = dim_product.groupby('product_id').size().reset_index(name='occurrences')
problem_products = product_dupes[product_dupes['occurrences'] > 1]

if len(problem_products) > 0:
    print(f"⚠️ Found {len(problem_products)} product_ids with multiple rows!")
    for _, row in problem_products.head(5).iterrows():
        prod_id = row['product_id']
        print(f"\nProduct ID {prod_id} appears {row['occurrences']} times:")
        display(dim_product[dim_product['product_id'] == prod_id])
else:
    print("✅ All product_ids are unique")

# Check if the same customer has inconsistent attributes
print("\n=== CONSISTENCY CHECK ===")
if len(problem_customers) > 0:
    sample_cust = problem_customers.iloc[0]['customer_id']
    print(f"Checking consistency for customer {sample_cust}:")
    cust_rows = dim_customer[dim_customer['customer_id'] == sample_cust]
    
    # Check if attributes are consistent
    if cust_rows['customer_name'].nunique() > 1:
        print(f"  ❌ Inconsistent customer_name: {cust_rows['customer_name'].tolist()}")
    if cust_rows['city'].nunique() > 1:
        print(f"  ❌ Inconsistent city: {cust_rows['city'].tolist()}")
    if cust_rows['country'].nunique() > 1:
        print(f"  ❌ Inconsistent country: {cust_rows['country'].tolist()}")

=== CUSTOMER DIMENSION ANALYSIS ===
⚠️ Found 2 customer_ids with multiple rows!

Customer ID C001 appears 2 times:


,customer_id,customer_name,city,country,valid_from,valid_to,is_current,customer_sk,source_files
0,C001,Ahmed Hassan,Cairo,Egypt,2025-01-02,2025-02-28,False,1,"sales_2025_01.csv, sales_2025_02.csv"
1,C001,Ahmed Hassan,Giza,Egypt,2025-03-01,NaT,True,2,sales_2025_03.csv



Customer ID C004 appears 2 times:


,customer_id,customer_name,city,country,valid_from,valid_to,is_current,customer_sk,source_files
4,C004,Mona Youssef,Mansoura,Egypt,2025-01-03,2025-03-02,False,5,sales_2025_01.csv
5,C004,Mona Youssef,Damietta,Egypt,2025-03-03,NaT,True,6,sales_2025_03.csv



=== PRODUCT DIMENSION ANALYSIS ===
⚠️ Found 11 product_ids with multiple rows!

Product ID P100 appears 3 times:


...


Product ID P109 appears 2 times:


,product_id,product_name,category,unit_price,unit_cost,valid_from,valid_to,is_current,product_sk,profit_margin,source_files
14,P109,Office Desk,Furniture,250.0,180.0,2025-01-06,2025-02-03,False,15,70.0,sales_2025_01.csv
15,P109,Office Desk,Furniture,255.0,180.0,2025-02-04,NaT,True,16,75.0,sales_2025_02.csv



Product ID P111 appears 2 times:


,product_id,product_name,category,unit_price,unit_cost,valid_from,valid_to,is_current,product_sk,profit_margin,source_files
17,P111,Router,Electronics,75.0,50.0,2025-01-07,2025-02-04,False,18,25.0,sales_2025_01.csv
18,P111,Router,Electronics,78.0,50.0,2025-02-05,NaT,True,19,28.0,sales_2025_02.csv



=== CONSISTENCY CHECK ===
Checking consistency for customer C001:
  ❌ Inconsistent city: ['Cairo', 'Giza']


In [193]:
# 6. Create Fact Table

In [194]:
# Step 6.1: Prepare Raw Data
print("=" * 60)
print("STEP 1: PREPARING RAW DATA")
print("=" * 60)

# Convert order_date to datetime if not already done
raw_data['order_date'] = pd.to_datetime(raw_data['order_date'])
print(f"✅ Raw data prepared: {len(raw_data)} rows")
print(f"📅 Date range: {raw_data['order_date'].min()} to {raw_data['order_date'].max()}")
print(f"📊 Columns: {list(raw_data.columns)}")

STEP 1: PREPARING RAW DATA
✅ Raw data prepared: 50 rows
📅 Date range: 2025-01-02 00:00:00 to 2025-03-05 00:00:00
📊 Columns: ['order_id', 'order_date', 'customer_id', 'customer_name', 'city', 'country', 'product_id', 'product_name', 'category', 'quantity', 'unit_price', 'unit_cost', 'source_file']


In [195]:
# Step 6.2: Join with Customer Dimension (SCD)

print("\n" + "=" * 60)
print("STEP 2: JOINING WITH CUSTOMER DIMENSION (TYPE 2 SCD)")
print("=" * 60)

# Join with customer dimension to get all possible versions
fact_sales = raw_data.merge(
    dim_customer[['customer_id', 'customer_sk', 'valid_from', 'valid_to']],
    on='customer_id',
    how='left'
)

print(f"📊 After customer join (before filtering): {len(fact_sales)} rows")

# Filter to the correct version based on order_date
before_filter = len(fact_sales)
fact_sales = fact_sales[
    (fact_sales['order_date'] >= fact_sales['valid_from']) & 
    ((fact_sales['valid_to'].isna()) | (fact_sales['order_date'] <= fact_sales['valid_to']))
]
after_filter = len(fact_sales)

print(f"✅ After filtering to correct version: {after_filter} rows")
print(f"   (Removed {before_filter - after_filter} rows with mismatched versions)")



STEP 2: JOINING WITH CUSTOMER DIMENSION (TYPE 2 SCD)
📊 After customer join (before filtering): 58 rows
✅ After filtering to correct version: 50 rows
   (Removed 8 rows with mismatched versions)


In [196]:
# Step 6.3: Join with Product Dimension (SCD)
print("\n" + "=" * 60)
print("STEP 3: JOINING WITH PRODUCT DIMENSION (TYPE 2 SCD)")
print("=" * 60)

# Join with product dimension
fact_sales = fact_sales.merge(
    dim_product[['product_id', 'product_sk', 'valid_from', 'valid_to']],
    on='product_id',
    how='left',
    suffixes=('_cust', '_prod')
)

print(f"📊 After product join (before filtering): {len(fact_sales)} rows")

# Filter to the correct product version based on order_date
before_filter = len(fact_sales)
fact_sales = fact_sales[
    (fact_sales['order_date'] >= fact_sales['valid_from_prod']) & 
    ((fact_sales['valid_to_prod'].isna()) | (fact_sales['order_date'] <= fact_sales['valid_to_prod']))
]
after_filter = len(fact_sales)

print(f"✅ After filtering to correct version: {after_filter} rows")
print(f"   (Removed {before_filter - after_filter} rows with mismatched versions)")


STEP 3: JOINING WITH PRODUCT DIMENSION (TYPE 2 SCD)
📊 After product join (before filtering): 86 rows
✅ After filtering to correct version: 50 rows
   (Removed 36 rows with mismatched versions)


In [197]:
# Step 6.4: Join with Date Dimension
print("\n" + "=" * 60)
print("STEP 4: JOINING WITH DATE DIMENSION")
print("=" * 60)

# Join with date dimension
before_join = len(fact_sales)
fact_sales = fact_sales.merge(
    dim_date[['date', 'date_sk']],
    left_on='order_date',
    right_on='date',
    how='left'
)
after_join = len(fact_sales)

print(f"✅ Date join complete: {after_join} rows")
print(f"   Date range in fact table: {fact_sales['date_sk'].min()} to {fact_sales['date_sk'].max()}")

# Check for any rows without date_sk
missing_dates = fact_sales[fact_sales['date_sk'].isna()]
if len(missing_dates) > 0:
    print(f"⚠️  Warning: {len(missing_dates)} rows have no matching date!")


STEP 4: JOINING WITH DATE DIMENSION
✅ Date join complete: 50 rows
   Date range in fact table: 1 to 25


In [198]:
# Step 6.5: Select and Validate Core Columns

print("\n" + "=" * 60)
print("STEP 5: SELECTING AND VALIDATING CORE COLUMNS")
print("=" * 60)

# Display available columns before selection
print("📋 Available columns before selection:")
print(list(fact_sales.columns))

# Select only the columns we need for the fact table
core_columns = ['order_id', 'date_sk', 'customer_sk', 'product_sk',
                'quantity', 'unit_price', 'unit_cost', 'source_file']

# Verify all required columns exist
missing_columns = [col for col in core_columns if col not in fact_sales.columns]
if missing_columns:
    print(f"❌ ERROR: Missing columns: {missing_columns}")
else:
    fact_sales = fact_sales[core_columns].copy()
    print(f"✅ Selected {len(core_columns)} core columns")
    print(f"   Fact table shape: {fact_sales.shape}")


STEP 5: SELECTING AND VALIDATING CORE COLUMNS
📋 Available columns before selection:
['order_id', 'order_date', 'customer_id', 'customer_name', 'city', 'country', 'product_id', 'product_name', 'category', 'quantity', 'unit_price', 'unit_cost', 'source_file', 'customer_sk', 'valid_from_cust', 'valid_to_cust', 'product_sk', 'valid_from_prod', 'valid_to_prod', 'date', 'date_sk']
✅ Selected 8 core columns
   Fact table shape: (50, 8)


In [199]:
# Step 6.6: Convert Data Types

print("\n" + "=" * 60)
print("STEP 6: CONVERTING DATA TYPES")
print("=" * 60)

# Track data types before conversion
print("📊 Data types before conversion:")
print(fact_sales.dtypes.to_string())

# Convert to appropriate types
fact_sales['order_id'] = fact_sales['order_id'].astype(str)
fact_sales['date_sk'] = fact_sales['date_sk'].astype(int)
fact_sales['customer_sk'] = fact_sales['customer_sk'].astype(int)
fact_sales['product_sk'] = fact_sales['product_sk'].astype(int)
fact_sales['quantity'] = fact_sales['quantity'].astype(int)
fact_sales['unit_price'] = fact_sales['unit_price'].astype(float)
fact_sales['unit_cost'] = fact_sales['unit_cost'].astype(float)
fact_sales['source_file'] = fact_sales['source_file'].astype(str)

print("\n✅ Data types after conversion:")
print(fact_sales.dtypes.to_string())


STEP 6: CONVERTING DATA TYPES
📊 Data types before conversion:
order_id        int64
date_sk         int64
customer_sk     int64
product_sk      int64
quantity        int64
unit_price      int64
unit_cost       int64
source_file    object

✅ Data types after conversion:
order_id        object
date_sk          int64
customer_sk      int64
product_sk       int64
quantity         int64
unit_price     float64
unit_cost      float64
source_file     object


In [200]:
# Step 6.7: Calculate Measures

print("\n" + "=" * 60)
print("STEP 7: CALCULATING MEASURES")
print("=" * 60)

# Calculate derived measures
fact_sales['total_sales'] = (fact_sales['quantity'] * fact_sales['unit_price']).round(2)
fact_sales['total_cost'] = (fact_sales['quantity'] * fact_sales['unit_cost']).round(2)
fact_sales['profit'] = (fact_sales['total_sales'] - fact_sales['total_cost']).round(2)

# Calculate profit margin with division by zero handling
# FIXED: Removed .round(2) from inside the lambda, will round after calculation
fact_sales['profit_margin'] = fact_sales.apply(
    lambda row: round((row['profit'] / row['total_sales'] * 100), 2) 
    if row['total_sales'] != 0 else 0, 
    axis=1
)

print(f"✅ Calculated measures:")
print(f"   - Total sales range: ${fact_sales['total_sales'].min():.2f} to ${fact_sales['total_sales'].max():.2f}")
print(f"   - Profit range: ${fact_sales['profit'].min():.2f} to ${fact_sales['profit'].max():.2f}")
print(f"   - Avg profit margin: {fact_sales['profit_margin'].mean():.2f}%")


STEP 7: CALCULATING MEASURES
✅ Calculated measures:
   - Total sales range: $24.00 to $1300.00
   - Profit range: $14.00 to $320.00
   - Avg profit margin: 38.70%


In [201]:
# Step 6.8: Final Fact Table Summary

print("\n" + "=" * 60)
print("STEP 8: FINAL FACT TABLE SUMMARY")
print("=" * 60)

print(f"📊 FINAL FACT TABLE STATISTICS:")
print(f"   - Total rows: {len(fact_sales):,}")
print(f"   - Unique orders: {fact_sales['order_id'].nunique():,}")
print(f"   - Unique products: {fact_sales['product_sk'].nunique():,}")
print(f"   - Unique customers: {fact_sales['customer_sk'].nunique():,}")
print(f"   - Date range: {fact_sales['date_sk'].min()} to {fact_sales['date_sk'].max()}")
print(f"   - Total sales: ${fact_sales['total_sales'].sum():,.2f}")
print(f"   - Total profit: ${fact_sales['profit'].sum():,.2f}")

# Verify no duplicates
duplicate_count = fact_sales.duplicated(subset=['order_id', 'product_sk']).sum()
print(f"\n✅ Duplicate check: {duplicate_count} duplicates found (should be 0)")

# Show sample
print("\n📋 Sample of final fact table (first 10 rows):")
display_cols = ['order_id', 'product_sk', 'quantity', 'total_sales', 'profit', 'profit_margin', 'source_file']
fact_sales[display_cols].head(10)


STEP 8: FINAL FACT TABLE SUMMARY
📊 FINAL FACT TABLE STATISTICS:
   - Total rows: 50
   - Unique orders: 50
   - Unique products: 38
   - Unique customers: 38
   - Date range: 1 to 25
   - Total sales: $10,742.00
   - Total profit: $3,038.00

✅ Duplicate check: 0 duplicates found (should be 0)

📋 Sample of final fact table (first 10 rows):


,order_id,product_sk,quantity,total_sales,profit,profit_margin,source_file
0,1001,1,1,1200.0,300.0,25.00,sales_2025_01.csv
1,1002,4,2,50.0,30.0,60.00,sales_2025_01.csv
2,1003,5,1,45.0,25.0,55.56,sales_2025_01.csv
3,1004,6,1,300.0,80.0,26.67,sales_2025_01.csv
4,1005,7,1,180.0,40.0,22.22,sales_2025_01.csv
5,1006,9,2,190.0,50.0,26.32,sales_2025_01.csv
6,1007,10,3,24.0,15.0,62.50,sales_2025_01.csv
7,1008,11,1,110.0,25.0,22.73,sales_2025_01.csv
8,1009,14,1,65.0,25.0,38.46,sales_2025_01.csv
9,1010,15,1,250.0,70.0,28.00,sales_2025_01.csv


In [202]:
# Step 6.9: Quality Checks

print("\n" + "=" * 60)
print("STEP 9: QUALITY CHECKS")
print("=" * 60)

# Check 1: Referential integrity - do all foreign keys exist?
print("🔍 Check 1: Referential Integrity")

# Customer check
missing_customers = fact_sales[~fact_sales['customer_sk'].isin(dim_customer['customer_sk'])]
print(f"   - Missing customer references: {len(missing_customers)} rows")

# Product check
missing_products = fact_sales[~fact_sales['product_sk'].isin(dim_product['product_sk'])]
print(f"   - Missing product references: {len(missing_products)} rows")

# Date check
missing_dates = fact_sales[~fact_sales['date_sk'].isin(dim_date['date_sk'])]
print(f"   - Missing date references: {len(missing_dates)} rows")

# Check 2: Data quality
print("\n🔍 Check 2: Data Quality")
print(f"   - Negative quantities: {(fact_sales['quantity'] < 0).sum()} rows")
print(f"   - Negative prices: {(fact_sales['unit_price'] < 0).sum()} rows")
print(f"   - Negative profit margins: {(fact_sales['profit_margin'] < 0).sum()} rows")
print(f"   - Zero total sales: {(fact_sales['total_sales'] == 0).sum()} rows")

# Check 3: Source file distribution
print("\n🔍 Check 3: Source File Distribution")
source_dist = fact_sales['source_file'].value_counts()
for source, count in source_dist.items():
    pct = count / len(fact_sales) * 100
    print(f"   - {source}: {count:,} rows ({pct:.1f}%)")


STEP 9: QUALITY CHECKS
🔍 Check 1: Referential Integrity
   - Missing customer references: 0 rows
   - Missing product references: 0 rows
   - Missing date references: 0 rows

🔍 Check 2: Data Quality
   - Negative quantities: 0 rows
   - Negative prices: 0 rows
   - Negative profit margins: 0 rows
   - Zero total sales: 0 rows

🔍 Check 3: Source File Distribution
   - sales_2025_01.csv: 20 rows (40.0%)
   - sales_2025_02.csv: 20 rows (40.0%)
   - sales_2025_03.csv: 10 rows (20.0%)


In [203]:
# Step 7. Creating the Database Schema

In [204]:
# table creation SQL (include SCD columns)
print("\n=== SQL FOR TABLE CREATION (WITH SCD) ===")

scd_sql = """
-- First, create the date dimension (needs to exist before fact table references it)
CREATE TABLE dim_date (
    date_sk INTEGER PRIMARY KEY,
    date DATE,
    year INTEGER,
    month INTEGER,
    day INTEGER,
    quarter INTEGER,
    day_of_week INTEGER,
    day_name VARCHAR(20),
    month_name VARCHAR(20),
    is_weekend BOOLEAN
);

-- Customer dimension with SCD
CREATE TABLE dim_customer (
    customer_sk INTEGER PRIMARY KEY,
    customer_id VARCHAR(50),
    customer_name VARCHAR(100),
    city VARCHAR(100),
    country VARCHAR(100),
    valid_from DATE,
    valid_to DATE,
    is_current BOOLEAN,
    source_files TEXT
);

-- Product dimension with SCD
CREATE TABLE dim_product (
    product_sk INTEGER PRIMARY KEY,
    product_id VARCHAR(50),
    product_name VARCHAR(200),
    category VARCHAR(100),
    unit_price DECIMAL(10,2),
    unit_cost DECIMAL(10,2),
    profit_margin DECIMAL(10,2),
    valid_from DATE,
    valid_to DATE,
    is_current BOOLEAN,
    source_files TEXT
);

-- Fact table with source attribution
CREATE TABLE fact_sales (
    order_id VARCHAR(50),
    date_sk INTEGER REFERENCES dim_date(date_sk),
    customer_sk INTEGER REFERENCES dim_customer(customer_sk),
    product_sk INTEGER REFERENCES dim_product(product_sk),
    quantity INTEGER,
    unit_price DECIMAL(10,2),
    unit_cost DECIMAL(10,2),
    total_sales DECIMAL(12,2),
    total_cost DECIMAL(12,2),
    profit DECIMAL(12,2),
    profit_margin DECIMAL(5,2),
    source_file VARCHAR(100),
    PRIMARY KEY (order_id, product_sk)  -- Changed: removed date_sk from PK
);
"""

print(scd_sql)


=== SQL FOR TABLE CREATION (WITH SCD) ===

-- First, create the date dimension (needs to exist before fact table references it)
CREATE TABLE dim_date (
    date_sk INTEGER PRIMARY KEY,
    date DATE,
    year INTEGER,
    month INTEGER,
    day INTEGER,
    quarter INTEGER,
    day_of_week INTEGER,
    day_name VARCHAR(20),
    month_name VARCHAR(20),
    is_weekend BOOLEAN
);

-- Customer dimension with SCD
CREATE TABLE dim_customer (
    customer_sk INTEGER PRIMARY KEY,
    customer_id VARCHAR(50),
    customer_name VARCHAR(100),
    city VARCHAR(100),
    country VARCHAR(100),
    valid_from DATE,
    valid_to DATE,
    is_current BOOLEAN,
    source_files TEXT
);

-- Product dimension with SCD
CREATE TABLE dim_product (
    product_sk INTEGER PRIMARY KEY,
    product_id VARCHAR(50),
    product_name VARCHAR(200),
    category VARCHAR(100),
    unit_price DECIMAL(10,2),
    unit_cost DECIMAL(10,2),
    profit_margin DECIMAL(10,2),
    valid_from DATE,
    valid_to DATE,
    is_curr

In [205]:
# === PART 2: CREATE DATABASE (with autocommit) ===
print("\n" + "=" * 60)
print("PART 1: CREATING DATABASE")
print("=" * 60)

from sqlalchemy import create_engine, text
import psycopg2
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT

DB_USER = 'source_user'
DB_PASSWORD = 'source_pass'
DB_HOST = 'postgres-source'
DB_PORT = '5432'
NEW_DB_NAME = 'sales_dw'

# Connect to default postgres database to create new database
print("Connecting to PostgreSQL server...")
conn = psycopg2.connect(
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database='postgres'  # Connect to default 'postgres' db
)
conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
cursor = conn.cursor()

# Check if database exists and drop/create
cursor.execute(f"SELECT 1 FROM pg_database WHERE datname = '{NEW_DB_NAME}'")
exists = cursor.fetchone()

if exists:
    print(f"Database '{NEW_DB_NAME}' exists. Terminating connections and dropping...")
    cursor.execute(f"""
        SELECT pg_terminate_backend(pid)
        FROM pg_stat_activity
        WHERE datname = '{NEW_DB_NAME}'
    """)
    cursor.execute(f"DROP DATABASE {NEW_DB_NAME}")
    print(f"✅ Dropped existing database '{NEW_DB_NAME}'")

cursor.execute(f"CREATE DATABASE {NEW_DB_NAME}")
print(f"✅ Created fresh database: {NEW_DB_NAME}")

cursor.close()
conn.close()

# --- Connect to the new database with SQLAlchemy ---
engine = create_engine(f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{NEW_DB_NAME}')

# Test connection
with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database()"))
    print(f"✅ Connected to: {result.scalar()}")

print("\n✅ Database setup complete!")


PART 1: CREATING DATABASE
Connecting to PostgreSQL server...
Database 'sales_dw' exists. Terminating connections and dropping...
✅ Dropped existing database 'sales_dw'
✅ Created fresh database: sales_dw
✅ Connected to: sales_dw

✅ Database setup complete!


In [206]:
# Execute the SQL to create tables
print("\n=== CREATING TABLES IN DATABASE ===")
with engine.begin() as conn:
    # Execute the SQL (split into separate statements)
    # First create dim_date
    conn.execute(text("""
        CREATE TABLE dim_date (
            date_sk INTEGER PRIMARY KEY,
            date DATE,
            year INTEGER,
            month INTEGER,
            day INTEGER,
            quarter INTEGER,
            day_of_week INTEGER,
            day_name VARCHAR(20),
            month_name VARCHAR(20),
            is_weekend BOOLEAN
        )
    """))
    print("✅ dim_date created")
    
    # Create dim_customer
    conn.execute(text("""
        CREATE TABLE dim_customer (
            customer_sk INTEGER PRIMARY KEY,
            customer_id VARCHAR(50),
            customer_name VARCHAR(100),
            city VARCHAR(100),
            country VARCHAR(100),
            valid_from DATE,
            valid_to DATE,
            is_current BOOLEAN,
            source_files TEXT
        )
    """))
    print("✅ dim_customer created")
    
    # Create dim_product
    conn.execute(text("""
        CREATE TABLE dim_product (
            product_sk INTEGER PRIMARY KEY,
            product_id VARCHAR(50),
            product_name VARCHAR(200),
            category VARCHAR(100),
            unit_price DECIMAL(10,2),
            unit_cost DECIMAL(10,2),
            profit_margin DECIMAL(10,2),
            valid_from DATE,
            valid_to DATE,
            is_current BOOLEAN,
            source_files TEXT
        )
    """))
    print("✅ dim_product created")
    
    # Create fact_sales with foreign keys
    conn.execute(text("""
        CREATE TABLE fact_sales (
            order_id VARCHAR(50),
            date_sk INTEGER REFERENCES dim_date(date_sk),
            customer_sk INTEGER REFERENCES dim_customer(customer_sk),
            product_sk INTEGER REFERENCES dim_product(product_sk),
            quantity INTEGER,
            unit_price DECIMAL(10,2),
            unit_cost DECIMAL(10,2),
            total_sales DECIMAL(12,2),
            total_cost DECIMAL(12,2),
            profit DECIMAL(12,2),
            profit_margin DECIMAL(5,2),
            source_file VARCHAR(100),
            PRIMARY KEY (order_id, product_sk)
        )
    """))
    print("✅ fact_sales created")

print("\n🎉 All tables created successfully!")


=== CREATING TABLES IN DATABASE ===
✅ dim_date created
✅ dim_customer created
✅ dim_product created
✅ fact_sales created

🎉 All tables created successfully!


In [46]:
# 8. Load to Database with Proper Schema

In [207]:
# 8. Load to Database with Proper Schema
from sqlalchemy import types

# Define custom type mappings for better control (UPDATED with SCD columns)
type_mapping = {
    # Customer dimension
    'customer_sk': types.Integer(),
    'customer_id': types.String(50),
    'customer_name': types.String(100),
    'city': types.String(100),
    'country': types.String(100),
    'valid_from': types.Date(),
    'valid_to': types.Date(),
    'is_current': types.Boolean(),
    'source_files': types.Text(),
    
    # Product dimension
    'product_sk': types.Integer(),
    'product_id': types.String(50),
    'product_name': types.String(200),
    'category': types.String(100),
    'unit_price': types.Numeric(10,2),
    'unit_cost': types.Numeric(10,2),
    'profit_margin': types.Numeric(10,2),
    'valid_from': types.Date(),
    'valid_to': types.Date(),
    'is_current': types.Boolean(),
    'source_files': types.Text(),
    
    # Date dimension
    'date_sk': types.Integer(),
    'date': types.Date(),
    'year': types.Integer(),
    'month': types.Integer(),
    'day': types.Integer(),
    'quarter': types.Integer(),
    'day_of_week': types.Integer(),
    'day_name': types.String(20),
    'month_name': types.String(20),
    'is_weekend': types.Boolean(),
    
    # Fact table
    'order_id': types.String(50),
    'quantity': types.Integer(),
    'unit_price': types.Numeric(10,2),
    'unit_cost': types.Numeric(10,2),
    'total_sales': types.Numeric(12,2),
    'total_cost': types.Numeric(12,2),
    'profit': types.Numeric(12,2),
    'profit_margin': types.Numeric(5,2),
    'source_file': types.String(100)
}

print("Creating tables with proper schema...")



# Load data with explicit type mapping
print("Loading dimension tables...")

# Load customer dimension (with ALL columns)
customer_columns = ['customer_sk', 'customer_id', 'customer_name', 'city', 'country', 
                    'valid_from', 'valid_to', 'is_current', 'source_files']
dim_customer[customer_columns].to_sql(
    'dim_customer', 
    engine, 
    if_exists='append', 
    index=False,
    dtype={col: type_mapping[col] for col in customer_columns if col in type_mapping}
)
print(f"  ✅ dim_customer loaded: {len(dim_customer)} rows")

# Load product dimension (with ALL columns)
product_columns = ['product_sk', 'product_id', 'product_name', 'category', 
                   'unit_price', 'unit_cost', 'profit_margin',
                   'valid_from', 'valid_to', 'is_current', 'source_files']
dim_product[product_columns].to_sql(
    'dim_product', 
    engine, 
    if_exists='append', 
    index=False,
    dtype={col: type_mapping[col] for col in product_columns if col in type_mapping}
)
print(f"  ✅ dim_product loaded: {len(dim_product)} rows")

# Load date dimension
date_columns = ['date_sk', 'date', 'year', 'month', 'day', 'quarter', 
                'day_of_week', 'day_name', 'month_name', 'is_weekend']
dim_date[date_columns].to_sql(
    'dim_date', 
    engine, 
    if_exists='append', 
    index=False,
    dtype={col: type_mapping[col] for col in date_columns if col in type_mapping}
)
print(f"  ✅ dim_date loaded: {len(dim_date)} rows")

# Load fact table
fact_columns = ['order_id', 'date_sk', 'customer_sk', 'product_sk', 'quantity', 
                'unit_price', 'unit_cost', 'total_sales', 'total_cost', 
                'profit', 'profit_margin', 'source_file']
fact_sales[fact_columns].to_sql(
    'fact_sales', 
    engine, 
    if_exists='append', 
    index=False,
    dtype={col: type_mapping[col] for col in fact_columns if col in type_mapping}
)
print(f"  ✅ fact_sales loaded: {len(fact_sales)} rows")

print("\n✅ All tables loaded successfully!")

# Verification
print("\n" + "=" * 60)
print("VERIFICATION")
print("=" * 60)

with engine.connect() as conn:
    # Check SCD columns exist
    print("\n📊 Checking SCD columns in customer dimension:")
    result = conn.execute(text("""
        SELECT column_name 
        FROM information_schema.columns 
        WHERE table_name = 'dim_customer'
        ORDER BY ordinal_position
    """))
    columns = [row[0] for row in result]
    print(f"   Columns: {', '.join(columns)}")
    
    # Check for SCD columns
    scd_columns = ['valid_from', 'valid_to', 'is_current', 'source_files']
    missing = [col for col in scd_columns if col not in columns]
    if missing:
        print(f"❌ Missing SCD columns: {missing}")
    else:
        print(f"✅ All SCD columns present")
    
    # Row counts
    print("\n📊 Row counts:")
    tables = ['dim_customer', 'dim_product', 'dim_date', 'fact_sales']
    for table in tables:
        result = conn.execute(text(f"SELECT COUNT(*) FROM {table}"))
        count = result.scalar()
        print(f"  {table}: {count:,} rows")

Creating tables with proper schema...
Loading dimension tables...
  ✅ dim_customer loaded: 38 rows
  ✅ dim_product loaded: 38 rows
  ✅ dim_date loaded: 25 rows
  ✅ fact_sales loaded: 50 rows

✅ All tables loaded successfully!

VERIFICATION

📊 Checking SCD columns in customer dimension:
   Columns: customer_sk, customer_id, customer_name, city, country, valid_from, valid_to, is_current, source_files
✅ All SCD columns present

📊 Row counts:
  dim_customer: 38 rows
  dim_product: 38 rows
  dim_date: 25 rows
  fact_sales: 50 rows


In [208]:
# 9. Verify the Data Warehouse with Data Types

In [209]:
print("\n" + "=" * 60)
print("📊 DATA WAREHOUSE VERIFICATION")
print("=" * 60)

# Check 1: Database Schema
print("\n🔍 CHECK 1: DATABASE SCHEMA")
print("-" * 40)

with engine.connect() as conn:
    # Get column information for all tables
    result = conn.execute(text("""
        SELECT 
            table_name, 
            column_name, 
            data_type,
            is_nullable
        FROM information_schema.columns 
        WHERE table_name IN ('dim_customer', 'dim_product', 'dim_date', 'fact_sales')
        ORDER BY table_name, ordinal_position
    """))
    
    current_table = ""
    for row in result:
        if row[0] != current_table:
            current_table = row[0]
            print(f"\n📋 {current_table.upper()}:")
        print(f"  - {row[1]}: {row[2]} (nullable: {row[3]})")

# Check 2: Row Counts
print("\n🔍 CHECK 2: ROW COUNTS")
print("-" * 40)

with engine.connect() as conn:
    tables = ['dim_customer', 'dim_product', 'dim_date', 'fact_sales']
    for table in tables:
        result = conn.execute(text(f"SELECT COUNT(*) FROM {table}"))
        count = result.scalar()
        print(f"  {table}: {count:>6,} rows")

# Check 3: Referential Integrity
print("\n🔍 CHECK 3: REFERENTIAL INTEGRITY")
print("-" * 40)

with engine.connect() as conn:
    # Check for orphaned fact records
    checks = [
        ("fact_sales", "customer_sk", "dim_customer"),
        ("fact_sales", "product_sk", "dim_product"),
        ("fact_sales", "date_sk", "dim_date")
    ]
    
    for fact_table, fk_column, dim_table in checks:
        result = conn.execute(text(f"""
            SELECT COUNT(*) 
            FROM {fact_table} f 
            LEFT JOIN {dim_table} d ON f.{fk_column} = d.{fk_column.split('_')[0] + '_sk'}
            WHERE d.{fk_column.split('_')[0] + '_sk'} IS NULL
        """))
        orphan_count = result.scalar()
        status = "✅" if orphan_count == 0 else "❌"
        print(f"  {status} {fact_table}.{fk_column} -> {dim_table}: {orphan_count} orphaned rows")

# Check 4: Data Quality Metrics
print("\n🔍 CHECK 4: DATA QUALITY METRICS")
print("-" * 40)

with engine.connect() as conn:
    # Check for negative values
    result = conn.execute(text("""
        SELECT 
            COUNT(CASE WHEN quantity < 0 THEN 1 END) as neg_quantity,
            COUNT(CASE WHEN unit_price < 0 THEN 1 END) as neg_price,
            COUNT(CASE WHEN total_sales < 0 THEN 1 END) as neg_sales,
            COUNT(CASE WHEN profit < 0 THEN 1 END) as neg_profit
        FROM fact_sales
    """))
    row = result.fetchone()
    print(f"  Negative quantities: {row[0]}")
    print(f"  Negative prices: {row[1]}")
    print(f"  Negative sales: {row[2]}")
    print(f"  Negative profits: {row[3]}")
    
    # Date range
    result = conn.execute(text("""
        SELECT 
            MIN(date) as earliest,
            MAX(date) as latest,
            COUNT(DISTINCT date) as unique_dates
        FROM dim_date
    """))
    row = result.fetchone()
    print(f"\n  Date range: {row[0]} to {row[1]}")
    print(f"  Unique dates: {row[2]}")

# Check 5: SCD Version Tracking
print("\n🔍 CHECK 5: SCD VERSION TRACKING")
print("-" * 40)

with engine.connect() as conn:
    # Check customer SCD versions
    result = conn.execute(text("""
        SELECT 
            COUNT(*) as total_rows,
            SUM(CASE WHEN is_current THEN 1 ELSE 0 END) as current_versions,
            COUNT(DISTINCT customer_id) as unique_customers
        FROM dim_customer
    """))
    row = result.fetchone()
    print(f"  Customer dimension:")
    print(f"    - Total rows: {row[0]}")
    print(f"    - Current versions: {row[1]}")
    print(f"    - Unique customers: {row[2]}")
    print(f"    - Avg versions per customer: {row[0]/row[2]:.1f}")
    
    # Check product SCD versions
    result = conn.execute(text("""
        SELECT 
            COUNT(*) as total_rows,
            SUM(CASE WHEN is_current THEN 1 ELSE 0 END) as current_versions,
            COUNT(DISTINCT product_id) as unique_products
        FROM dim_product
    """))
    row = result.fetchone()
    print(f"\n  Product dimension:")
    print(f"    - Total rows: {row[0]}")
    print(f"    - Current versions: {row[1]}")
    print(f"    - Unique products: {row[2]}")
    print(f"    - Avg versions per product: {row[0]/row[2]:.1f}")

# Check 6: Analytics Query
print("\n🔍 CHECK 6: SAMPLE ANALYTICS")
print("-" * 40)

query = """
SELECT 
    d.year,
    d.month,
    d.month_name,
    p.category,
    COUNT(DISTINCT f.order_id) as num_orders,
    SUM(f.quantity) as total_units,
    ROUND(SUM(f.total_sales)::numeric, 2) as revenue,
    ROUND(SUM(f.profit)::numeric, 2) as profit,
    ROUND(AVG(f.profit_margin)::numeric, 2) as avg_profit_margin
FROM fact_sales f
JOIN dim_date d ON f.date_sk = d.date_sk
JOIN dim_product p ON f.product_sk = p.product_sk
GROUP BY d.year, d.month, d.month_name, p.category
ORDER BY d.year, d.month, revenue DESC
LIMIT 10
"""

try:
    analytics = pd.read_sql(query, engine)
    print("\n📊 Analytics Sample (Top 10 rows):")
    print("-" * 40)
    
    if len(analytics) > 0:
        # Format the display
        display_cols = ['year', 'month', 'month_name', 'category', 'num_orders', 
                       'total_units', 'revenue', 'profit', 'avg_profit_margin']
        print(analytics[display_cols].to_string(index=False))
        
        # Summary stats
        print(f"\n  Total revenue in sample: ${analytics['revenue'].sum():,.2f}")
        print(f"  Total profit in sample: ${analytics['profit'].sum():,.2f}")
        print(f"  Avg profit margin: {analytics['avg_profit_margin'].mean():.2f}%")
    else:
        print("  No data returned from analytics query")
        
except Exception as e:
    print(f"❌ Analytics query failed: {e}")
    print("\nTrying simplified query...")
    
    # Fallback query if the first one fails
    simple_query = """
    SELECT 
        COUNT(*) as total_rows,
        COUNT(DISTINCT order_id) as unique_orders,
        ROUND(SUM(total_sales)::numeric, 2) as total_revenue,
        ROUND(SUM(profit)::numeric, 2) as total_profit,
        ROUND(AVG(profit_margin)::numeric, 2) as avg_margin
    FROM fact_sales
    """
    
    simple_stats = pd.read_sql(simple_query, engine)
    print("\n📊 Basic Statistics:")
    print(simple_stats.to_string(index=False))

# Check 7: Source File Distribution
print("\n🔍 CHECK 7: SOURCE FILE DISTRIBUTION")
print("-" * 40)

with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT 
            source_file,
            COUNT(*) as row_count,
            ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) as percentage
        FROM fact_sales
        GROUP BY source_file
        ORDER BY row_count DESC
    """))
    
    print("\n  Rows by source file:")
    for row in result:
        print(f"    - {row[0]}: {row[1]:,} rows ({row[2]}%)")

print("\n" + "=" * 60)
print("✅ VERIFICATION COMPLETE")
print("=" * 60)


📊 DATA WAREHOUSE VERIFICATION

🔍 CHECK 1: DATABASE SCHEMA
----------------------------------------

📋 DIM_CUSTOMER:
  - customer_sk: integer (nullable: NO)
  - customer_id: character varying (nullable: YES)
  - customer_name: character varying (nullable: YES)
  - city: character varying (nullable: YES)
  - country: character varying (nullable: YES)
  - valid_from: date (nullable: YES)
  - valid_to: date (nullable: YES)
  - is_current: boolean (nullable: YES)
  - source_files: text (nullable: YES)

📋 DIM_DATE:
  - date_sk: integer (nullable: NO)
  - date: date (nullable: YES)
  - year: integer (nullable: YES)
  - month: integer (nullable: YES)
  - day: integer (nullable: YES)
  - quarter: integer (nullable: YES)
  - day_of_week: integer (nullable: YES)
  - day_name: character varying (nullable: YES)
  - month_name: character varying (nullable: YES)
  - is_weekend: boolean (nullable: YES)

📋 DIM_PRODUCT:
  - product_sk: integer (nullable: NO)
  - product_id: character varying (nullable:

In [210]:
# Step 10: Export All Tables

print("\n" + "=" * 60)
print("STEP 10: EXPORTING ALL TABLES")
print("=" * 60)

output_path = '/home/jovyan/data/transformed/'
os.makedirs(output_path, exist_ok=True)

# Dictionary of dataframes to export
tables_to_export = {
    'dim_customer': dim_customer,
    'dim_product': dim_product,
    'dim_date': dim_date,
    'fact_sales': fact_sales
}

total_size = 0
for name, df in tables_to_export.items():
    filename = f'{output_path}{name}.csv'
    df.to_csv(filename, index=False)
    size = os.path.getsize(filename) / 1024
    total_size += size
    print(f"✅ {name}.csv: {len(df):,} rows, {size:.1f} KB")

print(f"\n📊 All tables exported to: {output_path}")
print(f"   Total size: {total_size:.1f} KB")
print(f"   Files: {', '.join(tables_to_export.keys())}.csv")


STEP 10: EXPORTING ALL TABLES
✅ dim_customer.csv: 38 rows, 2.8 KB
✅ dim_product.csv: 38 rows, 3.5 KB
✅ dim_date.csv: 25 rows, 1.3 KB
✅ fact_sales.csv: 50 rows, 3.3 KB

📊 All tables exported to: /home/jovyan/data/transformed/
   Total size: 10.8 KB
   Files: dim_customer, dim_product, dim_date, fact_sales.csv
